# ⚡ PySpark — Big Data Processing
## Python Ecosystem Tutorial Series — Module 12 of 18

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Library** | ⚡ PySpark |
| **Domain** | Big Data Processing |
| **Dataset** | 1M row chemical dataset |
| **Module** | 12 of 18 |

**What you will learn:**

1. What PySpark is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install pyspark
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `SparkSession.builder` | Create Spark session |
| `sdf.filter()` | Filter rows |
| `sdf.groupBy().agg()` | Group and aggregate |
| `F.col("name")` | Reference a column |
| `sdf.toPandas()` | Convert to pandas |

# 12. ⚡ PySpark — Big Data Processing
> **Python + PySpark = Big Data Processing**

PySpark is Python's interface to Apache Spark — processes **terabytes of data**
across clusters. Pandas handles millions of rows; Spark handles billions.

**Key concepts:** SparkSession, RDDs, DataFrames, transformations, actions, distributed SQL

In [ ]:
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import *
    from pyspark.ml.feature import VectorAssembler, StandardScaler
    from pyspark.ml.classification import RandomForestClassifier
    from pyspark.ml import Pipeline
    SPARK_OK = True
    spark = SparkSession.builder \
        .appName("PythonEcosystem_PySpark") \
        .config("spark.driver.memory", "2g") \
        .master("local[2]") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    print(f"PySpark version: {spark.version}")
except Exception as e:
    SPARK_OK = False
    print(f"PySpark not available: {e}")
    print("pip install pyspark")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Generate 1 million rows of synthetic chemical data ────────────────────────
np.random.seed(42)
N = 1_000_000
print(f"\nGenerating {N:,} rows of synthetic chemical data...")

df_big = pd.DataFrame({
    "chemical_id": range(N),
    "mw":          np.random.lognormal(5.2, 0.6, N).round(2),
    "logp":        np.random.normal(2.5, 1.8, N).round(3),
    "tpsa":        np.random.lognormal(3.8, 0.6, N).round(2),
    "hba":         np.random.poisson(3.5, N),
    "hbd":         np.random.poisson(1.8, N),
    "rotatable_bonds": np.random.poisson(4.5, N),
    "category":    np.random.choice(["drug","pesticide","industrial","food","cosmetic"], N,
                                     p=[0.3,0.15,0.25,0.20,0.10]),
})

# Lipinski Rule of 5 compliance
df_big["lipinski"] = ((df_big["mw"]<=500) & (df_big["logp"]<=5) &
                       (df_big["hba"]<=10) & (df_big["hbd"]<=5)).astype(int)

print(f"Dataset size: {len(df_big):,} rows, {df_big.memory_usage(deep=True).sum()/1e6:.1f} MB")

In [ ]:
# ── PySpark operations ────────────────────────────────────────────────────────
import time

if SPARK_OK:
    # Convert pandas → Spark DataFrame
    t0 = time.time()
    sdf = spark.createDataFrame(df_big)
    print(f"Spark DataFrame created in {time.time()-t0:.2f}s")
    print(f"Schema:")
    sdf.printSchema()

    # Lazy transformations (NOT executed yet)
    filtered = sdf.filter((F.col("mw").between(150, 500)) &
                            (F.col("logp").between(-2, 5)))
    grouped  = filtered.groupBy("category").agg(
        F.count("*").alias("count"),
        F.avg("mw").alias("avg_mw"),
        F.avg("logp").alias("avg_logp"),
        F.sum("lipinski").alias("lipinski_pass"),
    )
    # Action: triggers actual computation
    t0 = time.time()
    results = grouped.orderBy("count", ascending=False).toPandas()
    print(f"\nGroupBy computed in {time.time()-t0:.2f}s on {N:,} rows")
    print(results.round(3))

    # SQL interface (Spark supports full SQL!)
    sdf.createOrReplaceTempView("chemicals")
    top_drugs = spark.sql("""
        SELECT category, COUNT(*) as n,
               AVG(mw) as avg_mw,
               AVG(logp) as avg_logp,
               SUM(CASE WHEN lipinski=1 THEN 1 ELSE 0 END)*100.0/COUNT(*) as pct_lipinski
        FROM chemicals
        WHERE mw BETWEEN 100 AND 600
        GROUP BY category
        ORDER BY n DESC
    """).toPandas()
    print("\n── Spark SQL result ──")
    print(top_drugs.round(2))
else:
    # Pure pandas equivalent (same operations, same result, just slower at scale)
    print("\n── Running equivalent analysis with pandas (PySpark not installed) ──")
    t0 = time.time()
    filtered = df_big[(df_big["mw"].between(150,500)) & (df_big["logp"].between(-2,5))]
    results  = filtered.groupby("category").agg(
        count    = ("mw","count"),
        avg_mw   = ("mw","mean"),
        avg_logp = ("logp","mean"),
        lipinski_pass = ("lipinski","sum"),
    ).reset_index().sort_values("count",ascending=False)
    print(f"Pandas computation: {time.time()-t0:.2f}s on {N:,} rows")
    print(results.round(3))
    top_drugs = results.copy()

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cats = results["category"].values
counts_v = results["count"].values
cols5 = ["#3498DB","#27AE60","#E74C3C","#F1C40F","#8E44AD"]
axes[0].bar(cats, counts_v, color=cols5, alpha=0.85, edgecolor="white")
axes[0].set_title(f"Records per Category\n({N:,} total rows)", fontweight="bold")
axes[0].set_ylabel("Count"); axes[0].tick_params(axis="x",rotation=25)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"{int(x/1000)}k"))

axes[1].scatter(df_big.sample(3000)["logp"], df_big.sample(3000)["mw"],
                alpha=0.2, s=5, color="#3498DB")
axes[1].axvline(5, c="r", ls="--", lw=1.5, label="logP≤5")
axes[1].axhline(500, c="orange", ls="--", lw=1.5, label="MW≤500")
axes[1].set_xlabel("logP"); axes[1].set_ylabel("MW (Da)")
axes[1].set_title(f"3,000 sample points from\n{N:,} row dataset", fontweight="bold")
axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

lip_rate = df_big.groupby("category")["lipinski"].mean()
axes[2].bar(lip_rate.index, lip_rate.values*100, color=cols5, alpha=0.85, edgecolor="white")
axes[2].set_title("Lipinski Compliance by Category", fontweight="bold")
axes[2].set_ylabel("% Passing Rule of 5"); axes[2].tick_params(axis="x",rotation=25)
axes[2].grid(True, alpha=0.3, axis="y")
for i, v in enumerate(lip_rate.values):
    axes[2].text(i, v*100+0.3, f"{v:.0%}", ha="center", fontsize=8, fontweight="bold")

plt.suptitle(f"PySpark — Big Data: {N:,} rows processed", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pyspark_bigdata.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: PySpark

### Distributed Computing Architecture
Spark splits data across workers. In local mode (your laptop): across CPU cores. On a cluster: across machines. The API is identical — only the `master` setting changes.

```python
SparkSession.builder.master("local[4]")   # 4 CPU cores
SparkSession.builder.master("spark://cluster:7077")  # full cluster
```

### Lazy Evaluation — Why It Matters
Spark does NOT execute transformations immediately. It builds a logical plan, then optimises and executes it only when an action is called:

```python
df2 = df.filter(...)          # lazy: no computation
df3 = df2.groupBy(...).agg()  # lazy: no computation
df3.show()                     # ACTION: spark runs the optimised plan
```

This allows the Catalyst Optimiser to push filters down, reorder joins, and eliminate unnecessary columns before any data moves.

### Spark vs Pandas Comparison
```python
# Pandas
df[df["mw"] > 100].groupby("category")["mw"].mean()

# PySpark (very similar API!)
df.filter(F.col("mw") > 100).groupBy("category").agg(F.avg("mw"))
```

### Real Cheminformatics Scale
- ChEMBL: 2M+ compounds
- PubChem: 100M+ substances
- ToxCast/Tox21: 10K chemicals x 1000 assays
- These require Spark for efficient processing.


## ✅ Key Takeaways — ⚡ PySpark

1. PySpark uses lazy evaluation — transformations build a plan, actions execute it
2. The Catalyst Optimiser automatically improves your query plan before running
3. Spark SQL lets you query distributed data with standard SQL syntax
4. Use toPandas() only for the final small result — not for intermediate steps

---
*Next: Continue to Module 13 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [hgoelgithub.github.io](https://hgoelgithub.github.io)*